# Example 0.13 / 1: Work an Exercise: Advertising Campaign

Companion notebook for **0.13 Practice Exercises** in the AMPL lab Documentation, section "Work an Exercise: Advertising Campaign". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Work an Exercise: Advertising Campaign

Exercise 1-1(a) asks how to spend an advertising budget of \$1 million to reach the largest
audience. One minute of TV costs \$20,000 and reaches 1.8 million people; one magazine page costs
\$10,000 and reaches 1 million. At least 10 minutes of TV must be bought.

Write the formulation before the code. With $T$ minutes of TV and $M$ magazine pages, costs in
thousands of dollars and audience in millions:

$$
\max\ 1.8\,T + 1.0\,M \quad \text{subject to} \quad 20\,T + 10\,M \le 1000,\quad T \ge 10,\quad M \ge 0.
$$

Then translate each symbol into a declaration, with the numbers as data so later parts of the
exercise only change data. In the data and run fragments, `ampl` has already read the model, as
described on Running AMPL from Python (AMPL lab documentation, section 0.5).

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set MEDIA;                          # advertising media
param audience {MEDIA} > 0;         # millions of people reached per unit
param cost {MEDIA} > 0;             # thousands of dollars per unit
param min_buy {MEDIA} >= 0;         # minimum units to buy
param budget > 0;                   # thousands of dollars

var Buy {m in MEDIA} >= min_buy[m]; # units bought: TV minutes or magazine pages

maximize Total_Audience: sum {m in MEDIA} audience[m] * Buy[m];  # millions reached

subject to Budget: sum {m in MEDIA} cost[m] * Buy[m] <= budget;  # spending within budget

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["MEDIA"] = ["TV", "Magazine"]
ampl.param["audience"] = {"TV": 1.8, "Magazine": 1.0}
ampl.param["cost"] = {"TV": 20, "Magazine": 10}
ampl.param["min_buy"] = {"TV": 10, "Magazine": 0}
ampl.param["budget"] = 1000

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print(ampl.var["Buy"].to_dict())
print("audience:", ampl.obj["Total_Audience"].value())

HiGHS 1.14.0: optimal solution; objective 98
0 simplex iterations
0 barrier iterations
solved
{'Magazine': 80, 'TV': 10}
audience: 98.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set MEDIA := TV Magazine;
param:   audience  cost  min_buy :=
TV          1.8     20     10
Magazine    1.0     10      0;
param budget := 1000;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Buy, Total_Audience;

HiGHS 1.14.0: optimal solution; objective 98
0 simplex iterations
0 barrier iterations
Buy [*] :=
Magazine  80
      TV  10
;

Total_Audience = 98

